# TAE: Tabular AutoEncoder

A Tabular Autoencoder (TAE) is a specific subset of autoencoders designed explicitly to handle messy, heterogeneous tabular data (rows and columns) rather than sequential text or images.

## Step 1: Preprocessing the Tabular Features

* Continuous Features: Must be scaled using MinMaxScaler or StandardScaler.
* Categorical Features: One-hot encoded or passed through a small PyTorch nn.Embedding layer before entering the main encoder bottleneck.

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# 1. Define the Tabular Autoencoder Architecture
class TabularAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super(TabularAutoencoder, self).__init__()
        
        # Encoder: Squeezes tabular columns down to a tight bottleneck
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, latent_dim) # The Bottleneck (Latent Space / Embedding)
        )
        
        # Decoder: Attempts to reconstruct the exact original row features
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim),
            nn.Sigmoid() # Keeps outputs bounded if data is scaled between 0 and 1
        )
        
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return encoded, decoded

# 2. Mock Data (e.g., 1000 Spotify tracks with 10 scaled features)
# [danceability, energy, loudness_scaled, acousticness, tempo_scaled, ...]
mock_tracks = np.random.rand(1000, 10).astype(np.float32)
data_tensor = torch.tensor(mock_tracks)

# 3. Instantiate Model
INPUT_FEATURES = 10
EMBEDDING_SIZE = 4 # Compress 10 columns into a 4-dimensional track embedding
model = TabularAutoencoder(input_dim=INPUT_FEATURES, latent_dim=EMBEDDING_SIZE)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

# 4. Training Loop (Self-Supervised: Target is the Input itself!)
for epoch in range(50):
    optimizer.zero_grad()
    
    # Forward pass
    latent_embeddings, reconstructed_output = model(data_tensor)
    
    # Loss calculates how closely reconstructed data matches the original data
    loss = criterion(reconstructed_output, data_tensor)
    
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 10 == 0:
        print(f"Epoch [{epoch+1}/50], Reconstruction Loss: {loss.item():.4f}")

Epoch [10/50], Reconstruction Loss: 0.0823
Epoch [20/50], Reconstruction Loss: 0.0711
Epoch [30/50], Reconstruction Loss: 0.0639
Epoch [40/50], Reconstruction Loss: 0.0576
Epoch [50/50], Reconstruction Loss: 0.0509


## Step 2: Extract embeddings

Extract the tracks embeddings:

In [4]:
# Switch model to evaluation mode
model.eval()

with torch.no_grad():
    # Pass all 1,000 tracks through just the encoder
    spotify_track_embeddings, _ = model(data_tensor)

print("Original shape:", data_tensor.shape)       # torch.Size([1000, 10])
print("Embedding shape:", spotify_track_embeddings.shape) # torch.Size([1000, 4])

Original shape: torch.Size([1000, 10])
Embedding shape: torch.Size([1000, 4])


# Step 3: choose a row in dataset and find 3 most similar rows

Find the most similar rows in the embedding space using cosine distance.

In [6]:
import torch.nn.functional as F

# pick any row as the query
query_idx = 0
query_vec = spotify_track_embeddings[query_idx]

# cosine similarity between the query and every track embedding
# (similarity in [-1, 1]; cosine distance = 1 - similarity)
similarities = F.cosine_similarity(query_vec.unsqueeze(0), spotify_track_embeddings)

# mask out the query itself so it cannot match itself, then take the top 3
similarities[query_idx] = -1.0
top_scores, top_idx = torch.topk(similarities, k=3)

print(f"Query track #{query_idx}: embedding = {query_vec.numpy().round(3)}\n")
print("3 most similar tracks (by cosine distance in embedding space):")
for rank, (idx, sim) in enumerate(zip(top_idx.tolist(), top_scores.tolist()), start=1):
    print(
        f"  {rank}. track #{idx:<4d} "
        f"cosine_sim={sim:.4f}  cosine_dist={1.0 - sim:.4f}  "
        f"embedding={spotify_track_embeddings[idx].numpy().round(3)}"
    )

Query track #0: embedding = [-0.841  0.113 -1.03  -0.149]

3 most similar tracks (by cosine distance in embedding space):
  1. track #395  cosine_sim=0.9954  cosine_dist=0.0046  embedding=[-0.48   0.118 -0.657 -0.047]
  2. track #380  cosine_sim=0.9925  cosine_dist=0.0075  embedding=[-0.717  0.095 -0.789 -0.   ]
  3. track #720  cosine_sim=0.9885  cosine_dist=0.0115  embedding=[-0.863  0.187 -1.45  -0.244]
